In [1]:
import threading
import time
import random
from queue import Queue
from IPython.display import clear_output, display
import sys

In [2]:
# Shared Data Structures
latest_temperatures = {0: "--", 1: "--", 2: "--"}
temperature_averages = {0: "--", 1: "--", 2: "--"}

# Synchronization
queue = Queue()
lock = threading.RLock()
condition = threading.Condition(lock)

In [3]:
def simulate_sensor(sensor_id):
    """Simulate a temperature sensor that updates every second."""
    while True:
        temperature = random.randint(15, 40)
        with lock:
            latest_temperatures[sensor_id] = f"{temperature}°C"
            queue.put((sensor_id, temperature))
        time.sleep(1)  # Update every second

In [4]:
def process_temperatures():
    """Compute average temperature every 5 seconds."""
    sensor_data = {0: [], 1: [], 2: []}

    while True:
        while not queue.empty():
            sensor_id, temp = queue.get()
            with lock:
                sensor_data[sensor_id].append(temp)
                if len(sensor_data[sensor_id]) > 10:  # Keep last 10 readings
                    sensor_data[sensor_id].pop(0)
                avg_temp = sum(sensor_data[sensor_id]) / len(sensor_data[sensor_id])
                temperature_averages[sensor_id] = f"{avg_temp:.2f}°C"
        time.sleep(5)  # Update every 5 seconds


In [5]:
def display_temperatures():
    """Update the display in Jupyter Notebook."""
    while True:
        with lock:
            clear_output(wait=True)
            print("Current Temperatures:")
            print(f"Latest Temperatures: Sensor 0: {latest_temperatures[0]}  "
                  f"Sensor 1: {latest_temperatures[1]}  Sensor 2: {latest_temperatures[2]}")
            print(f"Sensor 0 Average: {temperature_averages[0]}")
            print(f"Sensor 1 Average: {temperature_averages[1]}")
            print(f"Sensor 2 Average: {temperature_averages[2]}\n")
        time.sleep(5)  # Refresh every 5 seconds


In [ ]:
# Start sensor threads
sensor_threads = [threading.Thread(target=simulate_sensor, args=(i,), daemon=True) for i in range(3)]
for thread in sensor_threads:
    thread.start()

# Start processing thread
processor_thread = threading.Thread(target=process_temperatures, daemon=True)
processor_thread.start()

# Start display thread
display_thread = threading.Thread(target=display_temperatures, daemon=True)
display_thread.start()

# Keep main thread alive
while True:
    time.sleep(1)

Current Temperatures:
Latest Temperatures: Sensor 0: 23°C  Sensor 1: 38°C  Sensor 2: 39°C
Sensor 0 Average: 26.60°C
Sensor 1 Average: 26.30°C
Sensor 2 Average: 29.90°C

